# Week 20 Optional: Advanced Monitoring Patterns

## Where this fits

You finished the main Week 20 notebook. The fraud system now has:

- Langfuse + Strands OTEL traces on `week19_supervisor`
- Data capture and a default-baseline Model Monitor schedule on `fraud-classifier-endpoint`
- A Spark-based PSI drift check
- A CloudWatch alarm on `ModelLatency`

This optional notebook extends three of those pieces. It is async. There are no in-class timings.

## What you will build

1. A LiteLLM-based inference path that auto-logs every LLM call to Langfuse without writing any tracing code yourself. This is the simpler alternative to Strands OTEL when you do not need an agent.
2. A CUSTOM `constraints.json` for Model Monitor that encodes business rules ("fraud rate must stay between 2% and 15%") instead of the auto-generated baseline.
3. The missing link from the main notebook: SNS topic + confirmed email subscription, so the CloudWatch alarm actually pages a human.

## Prerequisites

- The main Week 20 notebook ran cleanly end-to-end.
- The `aws-course-creds` secret scope still has all Week 20 keys.
- One new key added by the instructor: `course-instructor-email` (the email that will receive alerts).


## Environment setup

You are on the same Databricks Runtime 15.4 LTS ML cluster used in the main Week 20 notebook. Most pins come from the cluster libraries. We only install LiteLLM in-notebook here because it was not part of the Week 20 main install.

We pin `langfuse>=2.50,<3` deliberately. Langfuse Python SDK v3 broke the LiteLLM `success_callback = ["langfuse"]` integration (tracked in LiteLLM issue 13137). Staying on the v2 line keeps the five-line callback wiring working as documented.


In [ ]:
# Install LiteLLM in-notebook. Langfuse 2.x and boto3 are already on the cluster
# from the main Week 20 install, but we re-state the langfuse pin to be explicit.
%pip install --quiet "litellm>=1.50" "langfuse>=2.50,<3"
dbutils.library.restartPython()


In [ ]:
# Verify versions after kernel restart. Use importlib.metadata, never __version__.
from importlib.metadata import version
print("litellm:", version("litellm"))
print("langfuse:", version("langfuse"))
print("boto3:", version("boto3"))
print("sagemaker:", version("sagemaker"))


In [ ]:
# Same secret-scope pattern as the main Week 20 notebook. No getpass.
import os
import boto3

AWS_ACCESS_KEY_ID = dbutils.secrets.get(scope="aws-course-creds", key="aws-access-key-id")
AWS_SECRET_ACCESS_KEY = dbutils.secrets.get(scope="aws-course-creds", key="aws-secret-access-key")
AWS_SESSION_TOKEN = dbutils.secrets.get(scope="aws-course-creds", key="aws-session-token")
AWS_REGION = "us-east-1"

os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
os.environ["AWS_SESSION_TOKEN"] = AWS_SESSION_TOKEN
os.environ["AWS_REGION"] = AWS_REGION
os.environ["AWS_DEFAULT_REGION"] = AWS_REGION

# Langfuse keys. LiteLLM picks these up directly from os.environ.
os.environ["LANGFUSE_PUBLIC_KEY"] = dbutils.secrets.get(scope="aws-course-creds", key="langfuse-public-key")
os.environ["LANGFUSE_SECRET_KEY"] = dbutils.secrets.get(scope="aws-course-creds", key="langfuse-secret-key")
os.environ["LANGFUSE_HOST"] = dbutils.secrets.get(scope="aws-course-creds", key="langfuse-host")

# Carry-overs from main Week 20.
ENDPOINT_NAME = "fraud-classifier-endpoint"
BEDROCK_MODEL_ID = "us.anthropic.claude-3-haiku-20240307-v1:0"
BUCKET = dbutils.secrets.get(scope="aws-course-creds", key="course-s3-bucket")
INSTRUCTOR_EMAIL = dbutils.secrets.get(scope="aws-course-creds", key="course-instructor-email")

sagemaker_client = boto3.client("sagemaker", region_name=AWS_REGION)
cloudwatch = boto3.client("cloudwatch", region_name=AWS_REGION)
sns = boto3.client("sns", region_name=AWS_REGION)
s3 = boto3.client("s3", region_name=AWS_REGION)

print("Region:", AWS_REGION)
print("Endpoint:", ENDPOINT_NAME)
print("Bucket:", BUCKET)
print("Alert recipient:", INSTRUCTOR_EMAIL)


In [ ]:
# Fail loud if anything is missing before we wire up downstream pieces.

# 1. Endpoint still exists and is InService.
resp = sagemaker_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
assert resp["EndpointStatus"] == "InService", (
    f"Endpoint {ENDPOINT_NAME} is {resp['EndpointStatus']}. "
    "Ask your instructor to redeploy the Week 19 endpoint."
)
print("Endpoint:", resp["EndpointStatus"])

# 2. Bedrock LLM responds (LiteLLM will call this).
bedrock_runtime = boto3.client("bedrock-runtime", region_name=AWS_REGION)
try:
    bedrock_runtime.converse(
        modelId=BEDROCK_MODEL_ID,
        messages=[{"role": "user", "content": [{"text": "ping"}]}],
        inferenceConfig={"maxTokens": 10, "temperature": 0},
    )
    print("Bedrock LLM probe: ok")
except Exception as e:
    print("Ask your instructor to enable Bedrock access for", BEDROCK_MODEL_ID)
    raise

# 3. S3 bucket reachable.
s3.head_bucket(Bucket=BUCKET)
print("S3 bucket: ok")


## Part 1 - LiteLLM as a unified inference gateway with Langfuse callbacks

### Why a second observability path

In the main Week 20 notebook you wired Strands telemetry into Langfuse via OpenTelemetry. That works great when every LLM call goes through a Strands agent. In reality, most production teams also have plenty of non-agent code that calls LLMs directly: a batch summarization job, a one-off scoring script, an ad-hoc analyst notebook. You want those traced too, without rewriting them as agents.

LiteLLM solves this. It is a thin wrapper over 100+ LLM providers (OpenAI, Bedrock, Anthropic, Azure, Vertex, etc.) that gives you a single `completion()` call. The key feature for us today: LiteLLM has built-in callbacks. Two lines of configuration and every call gets logged to Langfuse automatically.

### The pattern

```python
import litellm
litellm.success_callback = ["langfuse"]
litellm.failure_callback = ["langfuse"]
```

LiteLLM reads `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, and `LANGFUSE_HOST` directly from `os.environ`. We already exported those in Cell 3.

### Why we pinned langfuse below v3

The LiteLLM Langfuse callback still depends on the langfuse v2 Python SDK. Langfuse 3.0 introduced architectural changes that broke the integration (LiteLLM issue 13137). The recommended workaround on v3 is `["langfuse_otel"]`, but that pattern has its own message-history rendering issues. For this notebook we stay on `langfuse>=2.50,<3` and use the stable `["langfuse"]` callback.


In [ ]:
# Five-line LiteLLM + Langfuse wiring. After this cell, every litellm.completion()
# call in this kernel is logged to your Langfuse project automatically.

import litellm

litellm.success_callback = ["langfuse"]
litellm.failure_callback = ["langfuse"]

# Single call. Note the `bedrock/` prefix on the model string. LiteLLM uses the
# AWS_* env vars from Cell 3 to authenticate to Bedrock.
resp = litellm.completion(
    model="bedrock/" + BEDROCK_MODEL_ID,
    messages=[
        {"role": "user", "content": "In one sentence, why does production ML need online monitoring?"}
    ],
    max_tokens=80,
    temperature=0,
    metadata={
        "generation_name": "week20-optional-demo",
        "tags": ["week20", "optional", "litellm-demo"],
    },
)
print(resp.choices[0].message.content)
print()
print("Open Langfuse -> Generations. You should see a new entry tagged week20.")


### Lab 1 - Batch fraud summary with LiteLLM + Langfuse

You are going to write a tiny batch job that summarizes a handful of suspicious transactions using LiteLLM. Every call will be auto-traced to Langfuse with a session id so you can group them in the UI.

Your task:

1. Read 5 rows from `bread_academy.course_data.fraud_transactions` where `is_fraud = 1`.
2. For each row, call `litellm.completion(...)` with `model="bedrock/" + BEDROCK_MODEL_ID` to produce a one-sentence summary.
3. Pass `metadata={"session_id": "fraud-batch-2026-05", "generation_name": "fraud-summary", "tags": ["week20", "optional", "lab1"]}` so all 5 calls land in the same Langfuse session.
4. Collect the summaries into a list called `summaries`.

Hints:
- Use `spark.read.table(...).filter("is_fraud = 1").limit(5).collect()` to get the rows.
- Build the user prompt from the row dict so the LLM has context.
- You do NOT need to re-set the success_callback in this lab; it is still set from the demo.

### Homework extension

After Lab 1 runs, open the Langfuse session view for `fraud-batch-2026-05` and write down the total prompt token count and total cost across the 5 calls. Compare with what the main Week 20 supervisor uses per query.


In [ ]:
# Lab 1 starter. The line after each YOUR CODE marker must NOT reveal the answer.

from pyspark.sql import functions as F

rows = None  # YOUR CODE

summaries = []
for row in rows or []:
    # YOUR CODE
    pass

print("Summaries collected:", len(summaries))
for s in summaries:
    print("-", s)


In [ ]:
# SAFETY-NET for Lab 1. Run only if `summaries` is still empty so the rest
# of the notebook still works. SKIP this cell if you finished Lab 1.

if not summaries:
    print("Using Lab 1 safety-net.")
    rows = (
        spark.read.table("bread_academy.course_data.fraud_transactions")
            .filter("is_fraud = 1")
            .limit(5)
            .collect()
    )
    for row in rows:
        prompt = (
            f"Summarize this suspicious transaction in one sentence: "
            f"amount=${row['transaction_amount']:.2f}, "
            f"merchant={row['merchant_category']}, "
            f"hours_since_last={row['hours_since_last_txn']:.1f}."
        )
        r = litellm.completion(
            model="bedrock/" + BEDROCK_MODEL_ID,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=80,
            temperature=0,
            metadata={
                "session_id": "fraud-batch-2026-05",
                "generation_name": "fraud-summary",
                "tags": ["week20", "optional", "lab1"],
            },
        )
        summaries.append(r.choices[0].message.content)
    print("Summaries collected via safety-net:", len(summaries))


## Part 2 - Custom Model Monitor constraints

### The limit of auto-baseline

In the main Week 20 notebook you called `monitor.suggest_baseline(...)` and let SageMaker infer statistics and constraints from the training CSV. That is fine for a first pass, but it has two problems:

1. The auto-baseline reflects whatever was in the training data, including any anomalies. If the training set had 50% fraud (because it was undersampled), the auto-baseline now expects 50% fraud in production. That is wrong.
2. There is no way to express business rules like "fraud rate must stay between 2% and 15%" or "average transaction amount must be between $50 and $500". Those are domain constraints, not statistical constraints.

### The fix: write your own constraints.json

A `constraints.json` file has three top-level keys: `version` (always `0`), `features` (per-feature constraints), and `monitoring_config` (global drift comparison settings). Each feature can carry a `num_constraints` block with `min`, `max`, and `is_non_negative`, plus a `completeness` floor. We will write a hand-tuned constraints file, upload it to S3, and point a new monitoring schedule at it.

### Reference schema

See https://docs.aws.amazon.com/sagemaker/latest/dg/model-monitor-byoc-constraints.html for the full schema. We use a minimal subset here.


In [ ]:
# Build a custom constraints.json that encodes business rules, upload it to S3,
# and have it sit next to a minimal statistics.json so Model Monitor can use both.

import json
from datetime import datetime

CUSTOM_PREFIX = f"fraud-classifier/monitoring/custom-baseline/{datetime.utcnow():%Y-%m-%d}"
constraints_key = f"{CUSTOM_PREFIX}/constraints.json"
statistics_key = f"{CUSTOM_PREFIX}/statistics.json"

custom_constraints = {
    "version": 0,
    "features": [
        {
            "name": "transaction_amount",
            "inferred_type": "Fractional",
            "completeness": 1.0,
            "num_constraints": {
                "is_non_negative": True,
                "min": 50.0,
                "max": 500.0,
            },
        },
        {
            "name": "hours_since_last_txn",
            "inferred_type": "Fractional",
            "completeness": 0.99,
            "num_constraints": {
                "is_non_negative": True,
                "min": 0.0,
                "max": 240.0,
            },
        },
        {
            "name": "merchant_category",
            "inferred_type": "String",
            "completeness": 1.0,
            "string_constraints": {
                "domains": [
                    "electronics", "groceries", "fuel",
                    "restaurants", "travel", "clothing",
                ],
            },
        },
    ],
    "monitoring_config": {
        "evaluate_constraints": "Enabled",
        "emit_metrics": "Enabled",
        "distribution_constraints": {
            "perform_comparison": "Enabled",
            "comparison_threshold": 0.1,
            "comparison_method": "Robust",
            "categorical_drift_method": "LInfinity",
        },
    },
}

# Minimal statistics.json. The schema mirrors what suggest_baseline emits; we
# include only what is required so the monitoring container does not error.
custom_statistics = {
    "version": 0,
    "dataset": {"item_count": 10000},
    "features": [
        {
            "name": "transaction_amount",
            "inferred_type": "Fractional",
            "numerical_statistics": {
                "common": {"num_present": 10000, "num_missing": 0},
                "mean": 240.0, "sum": 2400000.0,
                "std_dev": 90.0, "min": 50.0, "max": 500.0,
            },
        },
        {
            "name": "hours_since_last_txn",
            "inferred_type": "Fractional",
            "numerical_statistics": {
                "common": {"num_present": 9900, "num_missing": 100},
                "mean": 24.0, "sum": 240000.0,
                "std_dev": 30.0, "min": 0.0, "max": 240.0,
            },
        },
    ],
}

s3.put_object(
    Bucket=BUCKET, Key=constraints_key,
    Body=json.dumps(custom_constraints, indent=2).encode("utf-8"),
)
s3.put_object(
    Bucket=BUCKET, Key=statistics_key,
    Body=json.dumps(custom_statistics, indent=2).encode("utf-8"),
)
print("Custom constraints:", f"s3://{BUCKET}/{constraints_key}")
print("Custom statistics: ", f"s3://{BUCKET}/{statistics_key}")


### Lab 2 - Create a monitoring schedule that uses your custom constraints

You will create a SECOND monitoring schedule on the same endpoint. The main Week 20 schedule used auto-baseline; this one uses the custom `constraints.json` you uploaded above. Both can run side by side.

Your task:

1. Build a `DefaultModelMonitor` (same shape as in the main Week 20 lab).
2. Call `monitor.create_monitoring_schedule(...)` with:
   - `monitor_schedule_name="fraud-classifier-hourly-custom"`
   - `endpoint_input=ENDPOINT_NAME`
   - `statistics=f"s3://{BUCKET}/{statistics_key}"` (the S3 URI of your statistics file)
   - `constraints=f"s3://{BUCKET}/{constraints_key}"` (the S3 URI of your constraints file)
   - `output_s3_uri=f"s3://{BUCKET}/fraud-classifier/monitoring/custom-schedule-results"`
   - `schedule_cron_expression=CronExpressionGenerator.hourly()`
   - `enable_cloudwatch_metrics=True`
3. Print the schedule name so you can find it in the SageMaker console.

Hint: the `role` ARN is in the same secret scope under `sagemaker-execution-role-arn` (you used it in Week 20 Lab 2).

### Homework extension

Pick ONE feature in `constraints.json` and tighten the bounds (e.g. drop `transaction_amount` max from 500 to 300). Re-upload and create a third schedule. After 2 hours, compare the violation reports in S3 between the two schedules. Write down which constraint fired first and why.


In [ ]:
# Lab 2 starter. The line after each YOUR CODE marker must NOT reveal the answer.

from sagemaker import Session
from sagemaker.model_monitor import DefaultModelMonitor, CronExpressionGenerator

sm_session = Session()
role = dbutils.secrets.get(scope="aws-course-creds", key="sagemaker-execution-role-arn")

custom_monitor = None  # YOUR CODE

custom_schedule_name = None  # YOUR CODE

print("Custom schedule:", custom_schedule_name)


In [ ]:
# SAFETY-NET for Lab 2. Run only if `custom_schedule_name` is still None.

if custom_schedule_name is None:
    print("Using Lab 2 safety-net.")
    custom_monitor = DefaultModelMonitor(
        role=role,
        instance_count=1,
        instance_type="ml.m5.xlarge",
        volume_size_in_gb=20,
        max_runtime_in_seconds=1800,
        sagemaker_session=sm_session,
    )
    custom_schedule_name = "fraud-classifier-hourly-custom"
    custom_monitor.create_monitoring_schedule(
        monitor_schedule_name=custom_schedule_name,
        endpoint_input=ENDPOINT_NAME,
        output_s3_uri=f"s3://{BUCKET}/fraud-classifier/monitoring/custom-schedule-results",
        statistics=f"s3://{BUCKET}/{statistics_key}",
        constraints=f"s3://{BUCKET}/{constraints_key}",
        schedule_cron_expression=CronExpressionGenerator.hourly(),
        enable_cloudwatch_metrics=True,
    )
    print("Custom schedule created via safety-net:", custom_schedule_name)


## Part 3 - Actually paging a human: CloudWatch alarm -> SNS topic -> email

### What is missing from Week 20

In the main Week 20 notebook you created a CloudWatch alarm on `ModelLatency`. If you re-read that cell, you will notice it has an `AlarmActions=[sns_topic_arn]` line that pulls a pre-provisioned topic ARN from secrets. The instructor created that topic before class. You never saw how, and you never wired your own email to it. If a real-world alarm fires today, no one gets paged.

This part closes that loop:

1. Create (or re-use) an SNS topic named `week20-optional-alerts` in `us-east-1`.
2. Subscribe your own email address (`INSTRUCTOR_EMAIL`) to the topic with protocol `email`.
3. Confirm the subscription (you have to click a link in the email AWS sends).
4. Create a second CloudWatch alarm on `Invocation4XXErrors` that fires on 5 consecutive 4xx errors and publishes to the new topic.
5. Test by sending a deliberately malformed payload to the endpoint and waiting for the email.

### The subscription-confirmation gotcha

`sns.subscribe(...)` returns a `SubscriptionArn` whose value is literally the string `"pending confirmation"` until the recipient clicks the AWS confirmation link in their email. CloudWatch will publish to the topic regardless, but unconfirmed subscribers receive nothing. This trips up almost everyone on their first try. Watch for the confirmation email from `no-reply@sns.amazonaws.com`.


In [ ]:
# Create the SNS topic and subscribe the instructor email. Idempotent.

topic_resp = sns.create_topic(Name="week20-optional-alerts")
topic_arn = topic_resp["TopicArn"]
print("SNS topic ARN:", topic_arn)

# Subscribe the email. AWS sends a confirmation message to that address.
sub_resp = sns.subscribe(
    TopicArn=topic_arn,
    Protocol="email",
    Endpoint=INSTRUCTOR_EMAIL,
    ReturnSubscriptionArn=True,
)
print("Subscription ARN:", sub_resp["SubscriptionArn"])
print()
print("Check your inbox for a message from no-reply@sns.amazonaws.com.")
print("Click the AWS confirmation link or no alerts will reach you.")
print()
print("To verify after clicking, run:")
print("  sns.list_subscriptions_by_topic(TopicArn=topic_arn)")
print("The PendingConfirmation subscription should now show a real ARN.")


### Lab 3 - Wire a 4xx-error alarm to the SNS topic

You will create a CloudWatch alarm on the SageMaker endpoint's `Invocation4XXErrors` metric and route it through the SNS topic you just created. The main Week 20 notebook did this for `ModelLatency` against a pre-baked topic. Now you do it end to end with your own topic.

Your task:

1. Call `cloudwatch.put_metric_alarm(...)` with these parameters:
   - `AlarmName="fraud-classifier-4xx-errors"`
   - `MetricName="Invocation4XXErrors"`, `Namespace="AWS/SageMaker"`
   - `Statistic="Sum"`, `Period=60`, `EvaluationPeriods=5`
   - `Threshold=1.0`, `ComparisonOperator="GreaterThanOrEqualToThreshold"`
   - `Dimensions=[{"Name": "EndpointName", "Value": ENDPOINT_NAME}, {"Name": "VariantName", "Value": "AllTraffic"}]`
   - `TreatMissingData="notBreaching"`
   - `AlarmActions=[topic_arn]`
2. Store the alarm name in a variable called `alarm_name` for the safety-net check.
3. Print "Alarm created" so you know it ran.

Hint: this is a single boto3 call. You already imported `cloudwatch` in Cell 3.

### Homework extension

After the subscription is confirmed, deliberately trigger the alarm by invoking the endpoint 6 times with a clearly malformed JSON body (e.g. `Body=b"not json"`). Wait 5 minutes. You should receive an email. Take a screenshot of the email and the CloudWatch alarm in ALARM state for your portfolio.


In [ ]:
# Lab 3 starter. The line after each YOUR CODE marker must NOT reveal the answer.

alarm_name = None  # YOUR CODE

# YOUR CODE

print("Alarm created:", alarm_name)


## Think About It

Take 3 minutes to reflect on these cross-topic questions. They have no single right answer.

1. You now have THREE places that LLM calls get traced: Strands OTEL inside `week19_supervisor`, LiteLLM callbacks for direct calls, and (if you set it up) langfuse_otel for a future Langfuse v3 migration. Which one would you reach for first for a brand-new project at Bread Financial, and why? What is the tradeoff between flexibility and consistency?

2. The custom `constraints.json` in Lab 2 hard-coded `transaction_amount` to be between $50 and $500. Six months from now, a new product launch shifts genuine customer spend to a higher average. The Model Monitor schedule will start firing violations every hour. How do you decide whether to (a) widen the constraints, (b) retrain the model, or (c) both? Who at Bread Financial signs off on this decision?

3. SNS email subscriptions require manual confirmation. That is friction for a class but it is also a security feature: it prevents anyone with API access from spamming arbitrary addresses. In a real on-call rotation with 8 engineers, how would you automate the subscribe-and-confirm flow without weakening that security property? Hint: there is a different SNS protocol that does not require interactive confirmation.

## Wrap-up

In about 20 cells you closed three gaps from the main Week 20 notebook:

- LiteLLM gives you Langfuse traces on every LLM call WITHOUT writing tracing code, with one line per callback list.
- Custom `constraints.json` lets you encode business rules ("fraud rate stays between 2% and 15%") that auto-baseline cannot express.
- SNS plus confirmed email subscriptions turn a CloudWatch alarm from a passive console widget into something that actually pages a human.

## Further reading

- LiteLLM Langfuse integration: https://docs.litellm.ai/docs/observability/langfuse_integration
- LiteLLM issue 13137 (Langfuse v3 incompatibility): https://github.com/BerriAI/litellm/issues/13137
- SageMaker Model Monitor constraints schema: https://docs.aws.amazon.com/sagemaker/latest/dg/model-monitor-byoc-constraints.html
- CloudWatch put_metric_alarm boto3 reference: https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/cloudwatch/client/put_metric_alarm.html
- Langfuse Python v2-to-v3 upgrade path: https://langfuse.com/docs/observability/sdk/upgrade-path/python-v2-to-v3
